# Анализ данных — занятие 4
### Статистика для долей, множественные гипотезы и кластеризация

**Логика занятия:** почему обычные Z- и t-критерии не годятся для долей → критерий для доли и пример «лекарство или плацебо» → связанные выборки и матрица сопряжённости → проблема множественных гипотез и поправки → кластеризация: три семейства методов, `DBSCAN` и коэффициент силуэта.

## 1. Критерий для доли

Обычные Z- и t-критерии для среднего предполагают величину, которая может принимать значения от 0 до бесконечности. Доля живёт на отрезке от 0 до 1, поэтому для неё критерий устроен иначе.

Для доли используется **Z-критерий**:

$$Z = \frac{\hat{p} - p_0}{\sqrt{\dfrac{p_0(1-p_0)}{n}}}$$

Здесь `p̂` — доля, полученная в эксперименте, `p₀` — проверяемая (нулевая) вероятность, `n` — число наблюдений. Среднеквадратичное отклонение считается автоматически по вероятности: `σ₀ = √(p₀(1−p₀))`. При верной H₀ величина `Z` распределена нормально с нулевым средним и единичной дисперсией.

Двусторонний p-value: `p = 2·(1 − CDF(|Z|))`.

In [ ]:
import sys
!{sys.executable} -m pip install statsmodels

In [ ]:
import numpy as np
import scipy.stats as ss
import matplotlib.pyplot as plt

## 2. Пример: лекарство или плацебо

Стандартная процедура даёт выздоровление `p₀ = 0.9`. Новое лекарство помогло 93 пациентам из 100 (`p̂ = 0.93`). Нужно понять: это лекарство или случайность?

H₀ — настоящая доля выздоровления равна `0.9`. Считаем Z-статистику для доли и p-value.

In [ ]:
x = np.array([1]*93 + [0]*7)
xm = x.mean()          # p̂ = 0.93
m0 = 0.9               # проверяемая доля

s0 = np.sqrt(m0 * (1 - m0))   # СКО для доли
N = x.shape[0]

z = (xm - m0) / (s0 / np.sqrt(N))
pv = (1 - ss.norm.cdf(abs(z))) * 2
print(f'Z = {z:.4f}')
print(f'PV = {pv:.4f}')

Получилось `PV ≈ 0.32 > 0.05`, H₀ не отвергается: разница между `0.93` и `0.9` лежит в пределах погрешности. Это плацебо, а не лекарство.

## 3. Связанные выборки и матрица сопряжённости

Если опрашивают одних и тех же людей до и после события, выборки **связанные**. Пример: рейтинг Черчилля до и после Второй мировой войны (90 «за» / 10 «против» → 93 «за» / 7 «против»).

Интересуют только те, кто **поменял** мнение. Их раскладывают по матрице сопряжённости:

|  | после «за» | после «против» |
| --- | --- | --- |
| до «за» | a | b |
| до «против» | c | d |

Мнение поменяли ровно `b` и `c`. Проверяем гипотезу «мнение не изменилось» статистикой:

$$Z = \frac{b - c}{\sqrt{(b+c) - \dfrac{(b-c)^2}{n}}}$$

где `n = a+b+c+d` — общий объём выборки. При верной H₀ величина `Z` распределена нормально. Ячейки `a` и `d` (никогда не менявшие мнение) в расчёт не входят.

In [ ]:
a, b = 2, 5
c, d = 8, 85
n = a + b + c + d

z = (b - c) / np.sqrt((b + c) - (b - c)**2 / n)
pv = 2 * (1 - ss.norm.cdf(abs(z)))

print(f"Z-статистика: {z:.4f}")
print(f"PV: {pv:.4f}")

## 4. Множественные гипотезы

Если проверять сразу много гипотез, среди них всегда найдутся «ложно значимые». Это эффект **множественности гипотез**: при пороге `α = 0.05` примерно 5% верных H₀ случайно объявляются альтернативой.

Смоделируем «экстрасенсов»: 200 наборов по 100 случайных чисел. У первых 200 объектов связи нет (обычные люди), у вторых 200 чисел коррелируют с исходными («экстрасенсы»). Посчитаем p-value корреляции Пирсона для каждого.

In [ ]:
N, M = 200, 100
x = np.random.rand(N, M)
y = np.concatenate((np.random.rand(N, M), x + np.random.rand(N, M) * 5), axis=0)
x = np.concatenate((x, x), axis=0)

a = []
for i in range(N * 2):
    R, pv = ss.pearsonr(x[i, :], y[i, :])
    a.append(pv)
a = np.array(a)

normal, psychic = a[:N], a[N:]

Без поправок: при `α = 0.05` примерно 5% обычных людей мы ошибочно объявим экстрасенсами (ошибка I рода).

In [ ]:
alpha = 0.05
print('обычные, верно обычные :', (normal >  alpha).sum())
print('обычные, ложно экстрасенсы:', (normal <= alpha).sum())
print('экстрасенсы, найдены     :', (psychic <= alpha).sum(), 'из', N)

### Поправка Бонферрони

Самый простой способ — сравнить p-value не с `α`, а с `α / n`, где `n` — число гипотез. Это очень строгая поправка: она почти не пропускает ложных срабатываний, но может «пропустить» и настоящие.

In [ ]:
alpha = 0.05 / (N * 2)
print('обычные, верно обычные :', (normal >  alpha).sum())
print('обычные, ложно экстрасенсы:', (normal <= alpha).sum())
print('экстрасенсы, найдены     :', (psychic <= alpha).sum(), 'из', N)

### Поправка Бенджамини–Хохберга (FDR)

Поправка Бонферрони слишком завышает порог. **Бенджамини–Хохберг** использует **адаптивный порог**: p-value упорядочивают и идут по ряду, пока значение не окажется ниже порога, растущего вместе с номером. Это даёт баланс между пропущенными и ложными гипотезами.

В `statsmodels` функция `multipletests` возвращает скорректированные p-value, которые уже можно сравнивать с исходным `α`.

In [ ]:
from statsmodels.stats.multitest import multipletests

alpha = 0.05
rejects, pvcorr, _, _ = multipletests(a, alpha=alpha, method='fdr_bh')

print('обычные, верно обычные :', (pvcorr[:N] >  alpha).sum())
print('обычные, ложно экстрасенсы:', (pvcorr[:N] <= alpha).sum())
print('экстрасенсы, найдены     :', (pvcorr[N:] <= alpha).sum(), 'из', N)

## 5. Кластеризация

Вспомним типологию задач по типу данных и наличию учителя:

|  | категориальные | числовые |
| --- | --- | --- |
| **с учителем** | классификация | регрессия |
| **без учителя** | кластеризация | аппроксимация |

**Кластеризация** — разбиение точек на группы так, что внутри группы точки близки, а между группами — далеки. Имена групп условны: содержательной разницы между ними нет, поэтому задача сводится к «раскраске» множества точек.

Методы делятся на три семейства: **метрические** (нужно расстояние между точками), **модельные** (задана модель-распределение) и **иерархические** (последовательное деление или объединение).

## 6. DBSCAN: плотностная кластеризация

Идея `DBSCAN`: две точки попадают в один кластер, если их можно соединить ломаной, каждое звено которой короче `eps`. Если точку нельзя достичь такой ломаной, она объявляется шумом.

Считаем данные `files/src2.dat` и посмотрим на них.

In [ ]:
f = open('files/src2.dat')
d = []
for s in f:
    d.append([float(x) for x in s.split()])
f.close()
d = np.array(d)

plt.scatter(d[:, 0], d[:, 1])
plt.title('Исходные данные src2.dat')
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

clf = DBSCAN(eps=10)
c = clf.fit_predict(d)
plt.scatter(d[:, 0], d[:, 1], c=c)
plt.title('DBSCAN, eps = 10')
plt.show()

## 7. Подбор eps по коэффициенту силуэта

`eps` — гиперпараметр, от него полностью зависит результат. Есть два способа его подобрать:

- **внешняя экспертиза** — перебирать руками, пока картинка не станет «красивой»;
- **внутренняя экспертиза** — максимизировать числовой критерий. Самый известный — **коэффициент силуэта**.

Коэффициент силуэта окружает кластеры шарами: `a` — расстояние внутри кластера, `b` — расстояние между центрами. Когда шары пересекаются, силуэт падает; когда далеки — стремится к 1. Берём то `eps`, при котором силуэт максимален.

In [ ]:
from sklearn.metrics import silhouette_score

sopt, epsopt, copt = -1, -1, None
for eps in np.linspace(1, 20, 40):
    clf = DBSCAN(eps=eps)
    c = clf.fit_predict(d)
    if len(set(c)) > 1:
        s = silhouette_score(d, c)
        if s > sopt:
            sopt, epsopt, copt = s, eps, c

print('оптимальный eps:', round(epsopt, 3))
print('коэффициент силуэта:', round(sopt, 4))

plt.scatter(d[:, 0], d[:, 1], c=copt)
plt.title(f'DBSCAN, eps = {epsopt:.2f}, silhouette = {sopt:.3f}')
plt.show()

---

**Итог занятия 4.** Разобрали критерий для доли и для связанных выборок (матрица сопряжённости), проблему множественных гипотез и поправки Бонферрони и Бенджамини–Хохберга, а также начали кластеризацию: типология задач, семейства методов, алгоритм `DBSCAN` и подбор `eps` через коэффициент силуэта.